In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(df.shape)        
print(df.dtypes)

df.isnull().sum().sum() 

(7043, 21)
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object


np.int64(0)

In [13]:
print(df['Churn'].value_counts(normalize=True))

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True)

Churn
No     0.734215
Yes    0.265785
Name: proportion, dtype: float64


In [14]:
binary_cols = ['Partner','Dependents','PhoneService','PaperlessBilling','Churn']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

df = pd.get_dummies(df, columns=['Contract','PaymentMethod',
                                  'InternetService'], drop_first=True)

df['AvgMonthlySpend'] = df['TotalCharges'] / (df['tenure'] + 1)

df.drop('customerID', axis=1, inplace=True)

In [15]:
df = df.drop('customerID', axis=1, errors='ignore')

print(df.select_dtypes(include='object').columns.tolist())

['gender', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']


C:\Users\shrey\AppData\Local\Temp\ipykernel_10832\1699378266.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(df.select_dtypes(include='object').columns.tolist())


In [16]:
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})

df = pd.get_dummies(df, columns=['MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                                  'DeviceProtection', 'TechSupport',
                                  'StreamingTV', 'StreamingMovies'], 
                    drop_first=True)

print(df.select_dtypes(include='object').columns.tolist()) 

print(df.shape)

[]
(7032, 32)


In [17]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import shap

X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)


model = RandomForestClassifier(
    n_estimators=100, 
    random_state=42,
    class_weight='balanced'   
)
model.fit(X_train, y_train)

preds = model.predict(X_test)
print(classification_report(y_test, preds))
print("AUC:", roc_auc_score(y_test, model.predict_proba(X_test)[:,1]))

              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1033
           1       0.63      0.49      0.55       374

    accuracy                           0.79      1407
   macro avg       0.73      0.69      0.71      1407
weighted avg       0.78      0.79      0.78      1407

AUC: 0.8182529986385119
